# Tools

- The separation between an agent and a bot is its ability to take actions, perceive the output of the actions and react accordingly.
- React Agents use this pattern, and we will call these simply as agents.
- Actions an agent can take are defined by the tools that we provide it.
- Tools can allow agent to access data, execute tasks, even call our agents, transforming it from a passive language model to the coordinator of a much more capable system.

![](https://i.imgur.com/E99JcKg.png)

## Defining a tool in Langchain

In [1]:
# @tool decorator needed before the function

from dotenv import load_dotenv

load_dotenv()

True

In [2]:
from langchain.tools import tool

@tool
def square_root(x: float) -> float:
    """Calculate the square root of a number"""
    return x ** 0.5

In [3]:
@tool("square_root") # defining the name of the tool in the decorator
def tool1(x: float) -> float:
    """Calculate the square root of the number"""
    return x ** 0.5

In [4]:
@tool("square_root", description="calculate the square root of a number")
def tool1(x: float) -> float:
    return x ** 0.5;

In [7]:
tool1.invoke({"x": 467}) # This is how agent calls it as well

21.61018278497431

# Adding to Agents

In [11]:
from langchain.agents import create_agent

agent = create_agent(
    model = "gpt-5-nano",
    tools=[tool1],
    system_prompt="You are an arithmetic wizard. Use your tools to calculate thesquare root and square of any number."
)

In [12]:
from langchain.messages import HumanMessage

question = HumanMessage(content="What is the square root of 467?")

response = agent.invoke(
    {"messages": [question]}
)

print(response['messages'][-1].content)

√467 ≈ 21.61018278497431
Rounded to six decimals: 21.610183


In [16]:
from pprint import pprint

pprint(response['messages']) # Notice here is the tool_calls section with name of the tool followed by a message of the type ToolMessage before we get the AIMessage with the response
# Respons efrom the tool to the agaent is called the tool message.

[HumanMessage(content='What is the square root of 467?', additional_kwargs={}, response_metadata={}, id='9d4a8304-8fbe-4e39-92ca-55d3840bc845'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 791, 'prompt_tokens': 158, 'total_tokens': 949, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D6emNdvapjwZfYEDOtrr3iGa9AZHJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c38c7-04f3-7572-bb7e-68ccc2617220-0', tool_calls=[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_FroCPFFQ9bXvrcUdztmDPVdI', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 158, 'output_tokens': 791, 'total_tok

In [15]:
print(response['messages'][1].tool_calls)

[{'name': 'square_root', 'args': {'x': 467}, 'id': 'call_FroCPFFQ9bXvrcUdztmDPVdI', 'type': 'tool_call'}]


# Using Tools to Search the Web

In [17]:
from langchain.agents import create_agent

agent = create_agent(
    "gpt-5-nano"
)

In [23]:
from langchain.messages import HumanMessage

question = HumanMessage(content="Who is the Mayor of New-York?")

response = agent.invoke(
    {"messages": [question]}
)

In [24]:
print(response['messages'][-1].content)

Do you mean New York City or New York State?

- New York City: Eric Adams has been the mayor since January 1, 2022.
- New York State: There is no mayor (the state is governed by a governor). As of 2024, the governor is Kathy Hochul.

If you want the very latest status today, I can check for you.


# Add Web Search Tool

In [25]:
from langchain.tools import tool
from typing import Dict, Any
from tavily import TavilyClient

tavily_client = TavilyClient()

@tool
def web_search(query: str) -> Dict[str, Any]:
    """Search the web for information"""
    return tavily_client.search(query)

web_search.invoke("Who is the current mayor of New York?")

{'query': 'Who is the current mayor of New York?',
 'follow_up_questions': None,
 'answer': None,
 'images': [],
 'results': [{'url': 'https://www.nyc.gov/mayors-office',
   'title': "Welcome to the Mayor's Office",
   'content': 'Zohran Mamdani was sworn in as Mayor of the City of New York on January 1st, 2026. Prior to becoming Mayor, he represented the 36th New York State Assembly',
   'score': 0.8409005,
   'raw_content': None},
  {'url': 'https://www.bbc.com/news/articles/cly4kr8gzr2o',
   'title': 'Mamdani seals remarkable victory - but real challenges await',
   'content': "Zohran Mamdani, the newly elected mayor of New York City, is notable in many ways. He will become the city's youngest mayor since 1892, its",
   'score': 0.7539928,
   'raw_content': None},
  {'url': 'https://x.com/NYCMayor',
   'title': 'Mayor Zohran Kwame Mamdani (@NYCMayor) / Posts / X',
   'content': 'Mayor Zohran Kwame Mamdani (@NYCMayor) - Posts - Fighting every day to deliver a city that working New Yo

In [26]:
# Create the agent -> Pass the web_search tool to it -> Invoke the agent -> Compare the response with and without the tool

agent = create_agent(
    model="gpt-5-nano",
    tools=[web_search]
)

question = HumanMessage(content="Who is the current mayor of New York?")

response = agent.invoke(
    {"messages": [question]}
)

In [28]:
pprint(response['messages']) # Shows the Human, Tool and AIMessage

[HumanMessage(content='Who is the current mayor of New York?', additional_kwargs={}, response_metadata={}, id='29160c3b-0de9-479b-8a98-e56808d73498'),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 351, 'prompt_tokens': 133, 'total_tokens': 484, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D6furClhAm2HcfvUocUQ8uklQ3yX0', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019c3909-b369-7e52-974b-4524f694087e-0', tool_calls=[{'name': 'web_search', 'args': {'query': 'current mayor of New York City 2026'}, 'id': 'call_d4iZqSJydIChQzg7hFeFMrmA', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tok

In [29]:
pprint(response['messages'][-1])

AIMessage(content='Zohran Mamdani is the current mayor of New York City. He was sworn in on January 1, 2026. If you meant New York State governor instead, I can provide that as well.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 884, 'prompt_tokens': 829, 'total_tokens': 1713, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 832, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-D6fvBPfwhhCMpKsB5t0WI7wkopBis', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019c390a-05c5-7812-9394-5f29c6094257-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 829, 'output_tokens': 884, 'total_tokens': 1713, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {

# LangSmith

1. [Sample Trace](https://smith.langchain.com/public/82aa7bb2-a22c-4871-842e-bfd34930c9c6/r)

![](https://i.imgur.com/3CtTOlM.png)